# Phase 4-A — 真實 Snapshot 相容性探測（不消耗批次額度）

這個 notebook 在 Colab 上對真實 Snapshot `20261101_205431_06359faf7ef4_IBKR_FINAL_GATE_FULL` 執行 **不消耗額度的** Phase 4-A 檢查：

1. 掛載 Google Drive（唯讀使用），clone 已審查的 Agent commit，安裝 Core `e31c57b`。
2. 執行 `ResearchOnlyRunner.probe()`：governance preflight、Core 身分 (RO05)、Core 成本 profile、成本集一致性 (CO07)、以及 **只讀 metadata** 的 Snapshot 探測：
   - 讀 `final_gate_manifest.json`（小型 metadata 檔）並計算其 SHA256；
   - 讀 Parquet **footer**：Arrow schema 與 *timestamp 欄位* 的 row-group min/max。
3. 把探測結果寫入 `MyDrive/Quant_Research/strategy-research-agent/probes/`（write-once），並提供下載。

**不會做的事**
- 不消耗 `RA-IBKR_TSM-SMA520_SMOKE-0001` 的批次額度（`probe()` 沒有 consume 路徑）。
- 不解碼任何 OHLCV，不讀 Holdout 資料，不對 Parquet 做全檔 hash，不寫入 Snapshot 資料夾。
- 不 commit、不 push、不修改 governance。

執行完後請把下載的 `*_probe.json` 上傳回 Claude 對話。

In [ ]:
# --- 1. 設定 ---
AGENT_REPO = "https://github.com/hh4832/strategy-research-agent.git"
BRANCH     = "feature/phase4a-tsm-research-only"
CODE_PIN   = "ba8f23334f95060a7122226542e3110daa2e944b"     # 已推送的 Phase 4-A 整合 commit
CORE_SHA   = "e31c57b6c9dd599d808516ce6664628bdc79319e"     # Core Phase 4-A（SCOPED_GO, CI 42142535761）
RUN_NAME   = "20261101_205431_06359faf7ef4_IBKR_FINAL_GATE_FULL"
SNAPSHOT_ROOT = None    # None = 在 /content/drive/MyDrive 下以「完全相同的資料夾名稱」搜尋（必須恰好一個）；或填完整路徑
PROBE_DIR  = "/content/drive/MyDrive/Quant_Research/strategy-research-agent/probes"
WORKDIR    = "/content/phase4a_probe"

import json, os, pathlib, re, subprocess, sys
assert sys.version_info >= (3, 11), f"Python >= 3.11 required, found {sys.version}"
assert re.fullmatch(r"[0-9a-f]{40}", CODE_PIN) and re.fullmatch(r"[0-9a-f]{40}", CORE_SHA)
def sh(cmd, check=True, env=None):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, env=env)
    print(r.stdout[-4000:], r.stderr[-2000:] if r.returncode else "", sep="")
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed ({r.returncode}): {cmd.split('@')[0]}")
    return r
print("ok")

In [ ]:
# --- 2. 取得程式碼（匿名 clone）並安裝 pinned Core ---
clean_env = {k: v for k, v in os.environ.items() if k not in ("GITHUB_TOKEN", "GH_TOKEN")}
sh(f"rm -rf {WORKDIR}")
sh(f"git clone --quiet --branch {BRANCH} {AGENT_REPO} {WORKDIR}", env=clean_env)
sh(f"cd {WORKDIR} && git checkout --quiet {CODE_PIN}")
sh(f"cd {WORKDIR} && pip install -q -e '.[test,research]' "
   f"'market-research-core @ git+https://github.com/hh4832/market-research-core@{CORE_SHA}'", env=clean_env)
os.chdir(WORKDIR); sys.path.insert(0, f"{WORKDIR}/src")
from strategy_agent.adapters.mrc_phase4a import core_phase4a
native, costs = core_phase4a()
ident = native.core_code_identity()
print("Core identity:", ident)
assert ident["core_git_sha"] == CORE_SHA

In [ ]:
# --- 3. 掛載 Drive 並定位 Snapshot（只比對資料夾名稱，不讀任何資料檔）---
from google.colab import drive
drive.mount("/content/drive")
if SNAPSHOT_ROOT is None:
    hits, base = [], pathlib.Path("/content/drive/MyDrive")
    for dirpath, dirnames, _ in os.walk(base):
        depth = len(pathlib.Path(dirpath).relative_to(base).parts)
        if RUN_NAME in dirnames:
            hits.append(pathlib.Path(dirpath) / RUN_NAME)
        dirnames[:] = [d for d in dirnames if d != RUN_NAME and not d.startswith(".")] if depth < 5 else []
    print("matches:", hits)
    assert len(hits) == 1, "找不到或找到多個同名資料夾；請在第 1 格把 SNAPSHOT_ROOT 設為完整路徑"
    SNAPSHOT_ROOT = hits[0]
SNAPSHOT_ROOT = pathlib.Path(SNAPSHOT_ROOT)
assert SNAPSHOT_ROOT.name == RUN_NAME and SNAPSHOT_ROOT.is_dir(), SNAPSHOT_ROOT
print("snapshot root:", SNAPSHOT_ROOT)
print("top-level entries:", sorted(p.name for p in SNAPSHOT_ROOT.iterdir()))

In [ ]:
# --- 4. 不消耗額度的探測 ---
from strategy_agent.research_only.runner import ResearchOnlyRunner
CFG = pathlib.Path("configs/research_only/tsm_sma_5_20_smoke_2024_06_v1_1.json")
runner = ResearchOnlyRunner()
uses_before = runner.uses_path.read_bytes()
out = runner.probe(CFG, snapshot_root=SNAPSHOT_ROOT)
assert out["slot_consumed"] is False and runner.uses_path.read_bytes() == uses_before
print("STATUS:", out["status"])
print("preflight:", out["preflight"]["status"], [b["code"] for b in out["preflight"]["blockers"]])
for b in out["blockers"]:
    print(f"  {b['code']:28s} {b['message']}")
obs = out["probe"]["observed"]
print("\nmanifest:", json.dumps(obs.get("manifest"), indent=1, default=str)[:3000])
pq_ = obs.get("parquet", {})
print("\nparquet:", {k: pq_.get(k) for k in ("relative_path", "size", "num_rows", "num_row_groups", "arrow_schema",
                                             "timestamp_column", "timestamp_tz", "created_by")})
cand = [g for g in pq_.get("row_groups", []) if g.get("candidate")]
print("candidate row groups for 2024-05-29..2024-06-28:", json.dumps(cand, indent=1))

In [ ]:
# --- 5. 保存探測結果（write-once）並下載 ---
from datetime import datetime, timezone
from strategy_agent.research_only.runner import write_once, _dumps
stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
target = pathlib.Path(PROBE_DIR) / f"{stamp}_{CODE_PIN[:7]}_probe.json"
record = {"kind": "PHASE4A_SNAPSHOT_PROBE", "slot_consumed": False, "code_pin": CODE_PIN, "core_sha": CORE_SHA,
          "snapshot_root": str(SNAPSHOT_ROOT), "result": out}
sha = write_once(target, _dumps(record))
print(target, sha)
from google.colab import files
local = pathlib.Path("/content") / target.name
local.write_bytes(target.read_bytes())
files.download(str(local))